# Task 2A — Policy-Grounded Financial Compliance Triage
Prepare an auditable dataset from fictional internal policies/scenarios. The target
JSON contains `category`, `risk_level`, `recommended_action`, and `rationale`.
Only the supplied policy governs the answer; this is not legal advice or a real-law task.
Teacher: **openai/gpt-oss-120b via Groq**. Planned student: **Qwen/Qwen2.5-3B-Instruct**.
No student loading or training occurs in Task 2A.

The same seeded 160-ID plan now uses batches of **five**, 2,500 completion tokens
and **65-second** free-tier spacing. Resume revalidates prior raw responses and
requests only missing IDs, preserving previous valid examples and failed audit entries.
At least **120 cleaned** examples are required; **130+** is the preferred margin.
For 20 existing valid examples, at most 28 new five-example batches are planned;
allow roughly 30 minutes plus processing/retry time. No minimum yield is fabricated.

Run cells in order. Existing saved outputs remain historical evidence; rerun to
show the resumed result. Never delete the raw file or prior clean examples to resume.
Completed splits are frozen and reused without teacher calls.

## 1. Repository/dependency setup
Open from your GitHub repository in Colab. If no checkout is present, supply its
public GitHub URL. Install the existing requirements; no training dependencies are
added. Locally, start Jupyter from the repository root after installing requirements.

In [ ]:
import os
import subprocess
import sys
from importlib.util import find_spec
from pathlib import Path

try:
    IN_COLAB = find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

repo_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "task2_genai/prompts.py").is_file()
    ),
    None,
)
if repo_root is None:
    repo_url = input("Public GitHub repository HTTPS URL: ").strip()
    if not repo_url.startswith("https://github.com/"):
        raise ValueError("Supply the public GitHub repository URL")
    checkout = Path.cwd() / "CDAZZDEV-MLE-Poshitha-Malagala"
    if not checkout.exists():
        subprocess.run(["git", "clone", repo_url, str(checkout)], check=True)
    repo_root = checkout.resolve()
    if not (repo_root / "task2_genai/prompts.py").is_file():
        raise RuntimeError("Use a checkout containing the Task 2A implementation")
if IN_COLAB:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-r",
            str(repo_root / "requirements.txt"),
        ],
        check=True,
    )
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print("Repository:", repo_root)

## 2. Full teacher system prompt
The production prompt is saved in `task2_genai/teacher_system_prompt.txt`.

```text
You are generating a dataset for policy-grounded financial compliance triage.
Generate realistic but entirely fictional scenarios and target answers. These are
internal synthetic policies, not jurisdiction-specific law. Use ONLY the supplied
policy excerpt for each assignment; never invoke external regulations or provide
legal advice. Do not invent extra policy requirements or thresholds.

Return ONE JSON object with an "examples" array and no surrounding commentary.
Produce exactly one example per supplied assignment. Each object must have exactly:
id, topic, policy_excerpt, scenario, category, risk_level, recommended_action,
rationale. Copy the assignment's id, topic, category and policy_excerpt exactly.
risk_level must be "low", "medium" or "high". Construct a scenario consistent with
the assigned target risk; the final risk/action must follow the supplied policy.
The recommended_action must be actionable and follow the applicable internal rule.
The rationale must explicitly refer to the supplied policy and connect its trigger
to the scenario facts. Never add facts that are absent from the scenario or policy.
Do not mention that data is synthetic or generated in the answer fields.

Use anonymous roles such as "a retail customer" or "a small business", not personal
or company names. Do not include real customer data, names, account numbers, emails,
phone numbers, addresses, government identifiers or other PII. Use generic location
labels from the internal policy rather than naming countries or real restricted lists.
Transaction amounts may be fictional; size alone is not a risk trigger unless the
supplied policy explicitly says so.

Vary situation, sentence structure, evidence, customer type, transaction size,
scenario length, difficulty and ambiguity using the assignment hints. Include
borderline cases and uncertainty: explain when missing evidence requires review
rather than assuming wrongdoing. For ambiguous cases, provide enough facts for the
policy's uncertainty rule to apply. Avoid stereotyped, repeated or lightly renamed
templates, including templates from the avoidance examples. Do not simply paraphrase
the policy into a scenario. Give concrete operational facts and meaningful contrasts
between confirmed concerns, unresolved concerns and verified clearances.

Keep scenarios within the requested word range, actions concise, and rationales
approximately 25-55 words. Keep the exact supplied policy text even if another wording
would be shorter. All fields must be non-empty strings. Produce valid JSON only.
```

In [ ]:
from task2_genai.prompts import TEACHER_SYSTEM_PROMPT
from task2_genai.schemas import STUDENT_MODEL, TEACHER_MODEL

print("Assessment teacher:", TEACHER_MODEL)
print("Planned student (not loaded):", STUDENT_MODEL)
print(TEACHER_SYSTEM_PROMPT)

## 3. Synthetic policy taxonomy
Fourteen topics each define low/medium/high triggers and actions. Two equivalent
excerpt styles vary wording without changing policy semantics. Every example must
echo its assigned excerpt exactly; neither scenarios nor answers use real laws.

In [ ]:
import pandas as pd
from IPython.display import JSON, display

from task2_genai.policies import POLICIES

display(
    pd.DataFrame(
        [
            {
                "topic": policy.topic,
                "category": policy.category,
                "policy_excerpt": policy.excerpt(),
                "alternate_wording": policy.excerpt(1),
            }
            for policy in POLICIES
        ]
    )
)

## 4. Generation configuration and secure Groq setup
Use environment/Colab Secrets for `GROQ_API_KEY`; the key is never displayed.
Keep `GROQ_MODEL=openai/gpt-oss-120b` for the assessment. Resume is enabled by default.
Keep the same plan size/seed, prompt, taxonomy and teacher as the checkpoint; batch
size may safely change from ten to five. Import reloads pick up updated Task 2 code
without deleting runtime files. Update the checkout first.

Five examples use a 2,500-token completion budget and supported low reasoning effort.
Free-tier pacing spaces completed teacher requests by 65 seconds. Transport retries
are bounded and respect Retry-After; excessively long cooldowns fail without retrying
early. Token-truncated responses are explicit failures, preserved in raw audit.

In [ ]:
import getpass
import importlib
import json
import logging

task2_modules = {}
for name in ("schemas", "client", "dataset", "generation"):
    task2_modules[name] = importlib.reload(
        importlib.import_module(f"task2_genai.{name}")
    )
GroqTeacherClient = task2_modules["client"].GroqTeacherClient
TeacherError = task2_modules["client"].TeacherError
load_checkpoint = task2_modules["generation"].load_checkpoint
GenerationConfig = task2_modules["schemas"].GenerationConfig
PolicyExample = task2_modules["schemas"].PolicyExample

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
config = GenerationConfig(
    candidate_count=160, batch_size=5, minimum_clean=120, target_clean=130, seed=42
)
RUN_GENERATION = True
RESUME_GENERATION = True
FREE_TIER_PACING = True
DATA_DIR = repo_root / "task2_genai" / "data"
RAW_PATH = DATA_DIR / "raw_teacher_examples.jsonl"
CLEAN_PATH = DATA_DIR / "cleaned_examples.jsonl"
SUMMARY_PATH = DATA_DIR / "analysis_summary.json"
teacher_model = os.environ.get("GROQ_MODEL") or TEACHER_MODEL
prior_clean = (
    [
        PolicyExample.model_validate(json.loads(line))
        for line in CLEAN_PATH.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    if CLEAN_PATH.exists()
    else []
)
checkpoint = load_checkpoint(RAW_PATH, config) if RAW_PATH.exists() else None
existing_valid = (
    len(checkpoint.examples) if checkpoint is not None else len(prior_clean)
)
missing_before_resume = (
    checkpoint.missing_count
    if checkpoint is not None
    else config.candidate_count - existing_valid
)
print("existing_valid:", existing_valid)
print("missing_before_resume:", missing_before_resume)
splits_frozen = any(
    (DATA_DIR / name).exists()
    for name in ("train.jsonl", "validation.jsonl", "test.jsonl", "split_manifest.json")
)
client = None
if prior_clean and checkpoint is None and len(prior_clean) < config.minimum_clean:
    raise RuntimeError(
        "Restore the raw checkpoint to resume partial clean data; existing clean records are unchanged."
    )
if checkpoint is not None and prior_clean:
    recovered = {example.id: example for example in checkpoint.examples}
    if any(
        example.id not in recovered or example != recovered[example.id]
        for example in prior_clean
    ):
        raise RuntimeError(
            "Clean/checkpoint mismatch; no teacher calls or clean-file changes made."
        )
if RUN_GENERATION and not splits_frozen and missing_before_resume:
    if not os.environ.get("GROQ_API_KEY") and IN_COLAB:
        try:
            from google.colab import userdata

            secret = userdata.get("GROQ_API_KEY")
            if secret:
                os.environ["GROQ_API_KEY"] = secret
            secret = None
            configured_model = userdata.get("GROQ_MODEL")
            if configured_model:
                teacher_model = configured_model
        except Exception:
            print("Colab Secret unavailable; use hidden key input if needed.")
    if not os.environ.get("GROQ_API_KEY"):
        key = getpass.getpass("Groq API key (hidden; blank retains checkpoint only): ")
        if key:
            os.environ["GROQ_API_KEY"] = key
        key = None
    try:
        client = GroqTeacherClient(
            model=teacher_model,
            max_completion_tokens=2500,
            min_interval_seconds=65 if FREE_TIER_PACING else 0,
        )
    except TeacherError as error:
        print(str(error), "Previous valid examples are retained.")
print("Teacher model:", teacher_model, "| Student:", STUDENT_MODEL)
if teacher_model == STUDENT_MODEL:
    raise ValueError("Teacher and student must differ")
display(JSON(config.model_dump()))

## 5. Resume only missing assignments
Replay the original journal through current Pydantic/assignment checks. Compatible
legacy ten-example batches are supported. Existing successes are never requested
again or replaced; duplicate IDs are rejected. New batch IDs append monotonically,
leaving every old audit byte intact, including the original 429 failures.
The final valid pool is sorted by ID for deterministic merging.

In [ ]:
from task2_genai.generation import generate_examples

generation = None
clean_examples = list(prior_clean)
summary = (
    json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
    if SUMMARY_PATH.exists()
    else {}
)
if splits_frozen:
    print(
        "Existing splits are frozen; zero teacher calls. Clean data remains unchanged."
    )
elif client is not None and RUN_GENERATION and missing_before_resume:
    generation = generate_examples(
        client, config, raw_path=RAW_PATH, resume=RESUME_GENERATION
    )
elif checkpoint is not None:
    generation = checkpoint
    print(
        "Checkpoint recovered without teacher calls; unresolved IDs can be resumed later."
    )
elif not prior_clean:
    print("Generation skipped; no examples fabricated.")
if generation is not None:
    for name, count in generation.resume_summary.items():
        print(f"{name}: {count}")
else:
    print("existing_valid:", existing_valid)
    print("missing_before_resume:", missing_before_resume)
    print("newly_generated:", 0)
    print("final_valid:", existing_valid)

## 6. Local validation summary
Pydantic validates risks, required text, lengths and taxonomy. Generation also checks
IDs, exact supplied policy and assigned risk. Schema/assignment failures are rejected;
valid siblings survive. Structural checks do not prove semantic correctness: manually
review policy/action/rationale consistency before submitting or training.

In [ ]:
if generation is not None:
    summary = {
        "configuration": config.model_dump(),
        "teacher_model": teacher_model,
        "resume": generation.resume_summary,
        "validation": {
            "planned_candidates": config.candidate_count,
            "returned_items": generation.returned_count,
            "accepted_before_dedup": len(generation.examples),
            "rejected_records": len(generation.rejected),
            "missing_requested_ids": generation.missing_count,
            "failed_batches_in_audit": generation.failed_batches,
        },
        "rejections": generation.rejected,
    }
    display(JSON(summary["validation"]))
    if generation.rejected:
        display(pd.DataFrame(generation.rejected))
elif summary:
    display(JSON(summary.get("validation", {})))
else:
    print("No validation results yet; run or resume generation with credentials.")

## 7. Exact and near-duplicate filtering
Normalize case/Unicode/punctuation for exact scenario comparisons. Token sequence
similarity at **0.88 or above** flags near duplicates; retain the first record and
record removals/similarity. This lexical heuristic is intentionally lightweight:
inspect borderline pairs and distributions rather than claiming semantic deduplication.
Filtering is global before splitting, including across different policy topics.

In [ ]:
from task2_genai.dataset import deduplicate_examples, examples_frame, write_jsonl

if generation is not None:
    # Prefer already retained clean examples when a new scenario repeats one.
    prior_ids = {example.id for example in prior_clean}
    ordered = sorted(
        generation.examples,
        key=lambda example: (example.id not in prior_ids, example.id),
    )
    deduplication = deduplicate_examples(ordered)
    clean_examples = sorted(deduplication.examples, key=lambda example: example.id)
    summary["deduplication"] = deduplication.statistics
    summary["duplicate_removals"] = deduplication.removed
    write_jsonl(CLEAN_PATH, (example.model_dump() for example in clean_examples))
    SUMMARY_PATH.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
if summary.get("deduplication"):
    display(JSON(summary["deduplication"]))
    if summary.get("duplicate_removals"):
        display(pd.DataFrame(summary["duplicate_removals"]))
else:
    print("No deduplication results yet.")
frame = examples_frame(clean_examples)

## 8. Topic/category and risk-level diversity
Report measured distributions, not just planned coverage. No custom chart colors are specified.

In [ ]:
import matplotlib.pyplot as plt

if not frame.empty:
    display(frame["topic"].value_counts().rename("topic_count"))
    display(frame["category"].value_counts().rename("category_count"))
    display(frame["risk_level"].value_counts().rename("risk_count"))
    display(pd.crosstab(frame["topic"], frame["risk_level"]))
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    for column, axis in zip(["topic", "category", "risk_level"], axes):
        frame[column].value_counts().plot.bar(ax=axis, title=column, ylabel="Examples")
    plt.tight_layout()
    plt.show()
else:
    print("No clean records available to plot.")

## 9. Input/scenario length diversity
Lengths are words in the policy+scenario and scenario alone. Token lengths will be measured with the student tokenizer in Task 2B.

In [ ]:
if not frame.empty:
    display(frame[["input_words", "scenario_words"]].describe())
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    frame["input_words"].plot.hist(ax=axes[0], bins=15, title="Input length (words)")
    frame["scenario_words"].plot.hist(
        ax=axes[1], bins=15, title="Scenario length (words)"
    )
    axes[0].set_xlabel("Words")
    axes[1].set_xlabel("Words")
    plt.tight_layout()
    plt.show()

## 10. Hard minimum and preferred retained size
At least **120 clean** examples are required for splitting; aim for **130+**.
Both figures refer to actual validation/deduplication results. Keep partial records
and rerun resume for unresolved IDs; never fabricate a replacement or delete the audit.
If all 160 IDs are valid but deduplication leaves too few, inspect the repetition:
resume will not regenerate an already accepted ID to conceal the quality problem.

In [ ]:
dataset_ready = len(clean_examples) >= config.minimum_clean
preferred_target_reached = len(clean_examples) >= config.target_clean
print("Final clean examples:", len(clean_examples))
print("Hard minimum:", config.minimum_clean, "| Ready to split:", dataset_ready)
print(
    "Preferred clean target:",
    config.target_clean,
    "| Reached:",
    preferred_target_reached,
)
if not dataset_ready:
    print("Splitting is blocked. Retain current files and resume unresolved IDs.")

## 11. Reproducible 80/10/10 split
Fixed seed 42. Use topic+risk strata when feasible, then topic, then risk; small
holdouts cannot represent all 42 topic/risk cells. Proportional largest-remainder
allocation gives exact rounded sizes. Global deduplication precedes the split.
Saved IDs/hashes make the test set auditable. Existing split files are never
overwritten; keep the test set untouched until Task 2C evaluation.

In [ ]:
import hashlib

from task2_genai.dataset import normalize_scenario, save_splits, split_examples

manifest_path = DATA_DIR / "split_manifest.json"
if dataset_ready and manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    for name, expected_hash in manifest["sha256"].items():
        assert (
            hashlib.sha256((DATA_DIR / f"{name}.jsonl").read_bytes()).hexdigest()
            == expected_hash
        )
    print("Existing split hashes verified; test set unchanged.")
    print(
        "Exact counts:",
        manifest["counts"],
        "| Stratification:",
        manifest["stratification"],
    )
elif dataset_ready:
    split = split_examples(
        clean_examples, seed=config.seed, minimum_clean=config.minimum_clean
    )
    try:
        paths = save_splits(DATA_DIR, split)
        print("Exact counts:", split.counts, "| Stratification:", split.stratification)
        display(JSON(paths))
    except FileExistsError as error:
        print(str(error), "Existing artifacts remain unchanged.")
else:
    print("No splits generated: minimum clean size not reached.")
if dataset_ready and manifest_path.exists():
    examples_by_id = {example.id: example for example in clean_examples}
    split_ids = [
        {
            json.loads(line)["id"]
            for line in (DATA_DIR / f"{name}.jsonl")
            .read_text(encoding="utf-8")
            .splitlines()
        }
        for name in ("train", "validation", "test")
    ]
    split_scenarios = [
        {normalize_scenario(examples_by_id[identifier].scenario) for identifier in ids}
        for ids in split_ids
    ]
    for sets in (split_ids, split_scenarios):
        assert not (sets[0] & sets[1] or sets[0] & sets[2] or sets[1] & sets[2])
    print("ID/scenario split leakage: False")

## 12. Sample chat-format records
The assistant contains only the four target fields. Metadata IDs are not fed to
the student. Later use Qwen's `tokenizer.apply_chat_template`; no tokenizer, weights
or ChatML strings are loaded/generated in Task 2A.

In [ ]:
from task2_genai.dataset import chat_record

for example in clean_examples[:2]:
    display(JSON(chat_record(example)))
if not clean_examples:
    print("No real teacher examples available to display.")

## 13. Saved artifact paths
Download the real files from Colab's Files panel or copy them to your repository
after reviewing them. Raw JSONL holds batch envelopes; cleaned JSONL holds one
validated example per line. Split JSONL holds chat messages. Failed raw envelopes
contain safe categories and no invented replacements.

In [ ]:
artifact_names = [
    "raw_teacher_examples.jsonl",
    "cleaned_examples.jsonl",
    "analysis_summary.json",
    "train.jsonl",
    "validation.jsonl",
    "test.jsonl",
    "split_manifest.json",
]
display(
    pd.DataFrame(
        [
            {
                "artifact": name,
                "path": str(DATA_DIR / name),
                "exists": (DATA_DIR / name).exists(),
                "bytes": (DATA_DIR / name).stat().st_size
                if (DATA_DIR / name).exists()
                else None,
            }
            for name in artifact_names
        ]
    )
)

## 14. Limitations and next phases
Pydantic proves structure, not the correctness of teacher reasoning. Review a
stratified sample and all flagged/rejected records for grounding, realism and PII
before submission. Lexical deduplication can miss paraphrases or remove similar
legitimate cases; inspect its removals and resulting coverage. Teacher output bias
can carry into the student. This small dataset is not a production compliance system
and must not be used as legal advice.

Groq quotas/network availability may reduce yield; the notebook reports actual counts
and does not invent examples. Commit/download actual generated datasets and the
executed notebook as evidence. Qwen's model card specifies a Qwen research license;
review its terms before Task 2B use. Task 2B training/baseline comparisons and Task 2C
held-out evaluation are pending. Tasks 1 and 3 are unchanged.